<a href="https://colab.research.google.com/github/yashborhade-gif/221A019_NLP_EXPRRIMENTS/blob/main/Speech_to_Text_NLP_Mini_assi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Speech-to-Text Transcription Using NLP

**NLP Mini Project**  
This Google Colab notebook records speech through the browser microphone and converts it into text using Google Speech Recognition.

**Requirements:** Internet connection and permission to use your microphone.

## Step 1: Install Required Libraries
Run this cell once.

In [ ]:
!pip -q install SpeechRecognition

## Step 2: Record Audio from the Browser Microphone
Run the cell and click **Record**. Speak clearly, then click **Stop**. The recorded audio is saved as a WAV file.

In [ ]:
from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
from pathlib import Path

RECORDING_HTML = r'''<div>
  <button id="record" style="font-size:16px;padding:10px 18px;">🎙️ Record</button>
  <button id="stop" disabled style="font-size:16px;padding:10px 18px;">⏹ Stop</button>
  <span id="status" style="margin-left:10px;">Ready</span>
  <audio id="player" controls style="display:block;margin-top:12px;width:100%;"></audio>
</div>
<script>
let recorder, chunks = [], stream;
const recordBtn = document.getElementById('record');
const stopBtn = document.getElementById('stop');
const status = document.getElementById('status');
const player = document.getElementById('player');
recordBtn.onclick = async () => {
  try {
    stream = await navigator.mediaDevices.getUserMedia({audio:true});
    recorder = new MediaRecorder(stream);
    chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.onstop = async () => {
      const blob = new Blob(chunks, {type: recorder.mimeType});
      const reader = new FileReader();
      reader.onloadend = () => {
        const base64 = reader.result.split(',')[1];
        google.colab.kernel.invokeFunction('notebook.save_audio', [base64], {});
      };
      reader.readAsDataURL(blob);
      player.src = URL.createObjectURL(blob);
      status.textContent = 'Recording saved';
      stream.getTracks().forEach(track => track.stop());
    };
    recorder.start();
    recordBtn.disabled = true;
    stopBtn.disabled = false;
    status.textContent = 'Recording… speak now';
  } catch (err) { status.textContent = 'Microphone permission denied or unavailable'; }
};
stopBtn.onclick = () => {
  if (recorder && recorder.state !== 'inactive') recorder.stop();
  recordBtn.disabled = false;
  stopBtn.disabled = true;
};
</script>'''

from google.colab import output
import base64

def save_audio(base64_audio):
    audio_bytes = base64.b64decode(base64_audio)
    Path('/content/recording.webm').write_bytes(audio_bytes)
    print('Audio recording saved to /content/recording.webm')

output.register_callback('notebook.save_audio', save_audio)
display(HTML(RECORDING_HTML))

## Step 3: Convert Speech into Text
After recording and stopping, run this cell. The audio is converted to WAV format and transcribed. Change `en-IN` to another supported language code if needed.

In [ ]:
!ffmpeg -y -i /content/recording.webm -ac 1 -ar 16000 /content/recording.wav -loglevel error

import speech_recognition as sr

recognizer = sr.Recognizer()
audio_file = sr.AudioFile('/content/recording.wav')

with audio_file as source:
    audio_data = recognizer.record(source)

try:
    recognized_text = recognizer.recognize_google(
        audio_data,
        language='en-IN'
    )
    print('Speech-to-Text Output')
    print('---------------------')
    print(recognized_text)
except sr.UnknownValueError:
    print('Sorry, the speech could not be understood. Please record again.')
except sr.RequestError as e:
    print('Recognition service error:', e)

Sorry, the speech could not be understood. Please record again.


## Sample Output

If you say: **Hello, my name is Yash. This is my NLP mini project.**

The output should be similar to:
```text
Speech-to-Text Output
---------------------
Hello, my name is Yash. This is my NLP mini project.
```

Actual output depends on your recorded speech and recognition accuracy.

In [ ]:

# ============================================
# NLP MINI PROJECT: SPEECH TO TEXT
# GOOGLE COLAB WORKING CODE
# ============================================

!pip -q install SpeechRecognition

from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import speech_recognition as sr
import subprocess

# STEP 1: Record voice using browser microphone

display(Javascript("""
async function recordAudio() {
    const stream = await navigator.mediaDevices.getUserMedia({
        audio: true
    });

    const recorder = new MediaRecorder(stream);
    let chunks = [];

    recorder.ondataavailable = event => {
        chunks.push(event.data);
    };

    recorder.start();

    // Record for 8 seconds
    await new Promise(resolve => setTimeout(resolve, 8000));

    recorder.stop();

    await new Promise(resolve => {
        recorder.onstop = resolve;
    });

    stream.getTracks().forEach(track => track.stop());

    const blob = new Blob(chunks, {
        type: recorder.mimeType
    });

    const reader = new FileReader();

    return await new Promise(resolve => {
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
    });
}
"""))

print("Recording will start now.")
print("Please speak clearly for 8 seconds.")

audio_data = eval_js("recordAudio()")

# STEP 2: Save recorded audio

audio_bytes = b64decode(audio_data.split(",")[1])

with open("/content/recording.webm", "wb") as f:
    f.write(audio_bytes)

print("Recording completed!")

# STEP 3: Convert WebM to WAV

subprocess.run([
    "ffmpeg",
    "-y",
    "-i", "/content/recording.webm",
    "-ac", "1",
    "-ar", "16000",
    "/content/recording.wav",
    "-loglevel", "error"
], check=True)

# STEP 4: Load audio

recognizer = sr.Recognizer()

with sr.AudioFile("/content/recording.wav") as source:
    audio = recognizer.record(source)

# STEP 5: Speech recognition

try:

    text = recognizer.recognize_google(
        audio,
        language="en-IN"
    )

    print("\n========== OUTPUT ==========")
    print("Speech-to-Text Transcription")
    print("============================")
    print(text)

except sr.UnknownValueError:

    print("Speech not recognized. Please speak clearly.")

except sr.RequestError as e:

    print("Internet or API error:", e)


<IPython.core.display.Javascript object>

Recording will start now.
Please speak clearly for 8 seconds.
Recording completed!
Speech not recognized. Please speak clearly.
